# タクシー運賃予測のための最初の BigQuery ML モデル

**学習目標**
  * 適切な BigQuery ML モデルタイプを選択し、オプションを指定する
  * ML モデルのパフォーマンスを評価する
  * データ品質のクリーンアップによってモデルのパフォーマンスを向上させる
  * SQL を使用してディープニューラルネットワーク（DNN）を作成する

## 概要
このノートブックでは、BigQuery ML を使用してタクシー運賃予測のための最初のモデルを構築します。BigQuery ML は、大規模な構造化データおよび半構造化データに対して ML モデルを高速に構築する方法を提供します。まず、BigQuery で作成するすべてのモデルを保持するためのデータセットを作成することから始めます。

### 環境変数の設定

In [1]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION

%load_ext google.cloud.bigquery

env: PROJECT=dsparing-sandbox
env: BUCKET=dsparing-sandbox
env: REGION=us-central1


## BigQuery データセットと Google Cloud Storage バケットの作成

BigQuery データセットは、BigQuery ML で構築されたテーブル、ビュー、モデルのコンテナです。以前のラボでまだ作成していない場合は、__serverlessml__ という名前のデータセットを作成しましょう。プロジェクト用の GCS バケットについても同様に行います。

In [2]:
%%bash

# serverlessml の BigQuery データセットが存在しない場合は作成
datasetexists=$(bq ls -d | grep -w serverlessml)

if [ -n "$datasetexists" ]; then
    echo -e "BigQuery dataset already exists, let's not recreate it."
else
    echo "Creating BigQuery dataset titled: serverlessml"

    bq --location=US mk --dataset \
        --description 'Taxi Fare' \
        $PROJECT:serverlessml
    echo "\nHere are your current datasets:"
    bq ls
fi    

# GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gcloud storage ls
fi

BigQuery dataset already exists, let's not recreate it.
Bucket exists, let's not recreate it.


## モデル 1: 生データ

生データのみを使用してモデルを構築してみましょう。あまり良い結果にはなりませんが、実際にこれを体験してみるのも時には有益です。

モデルの訓練には 1 分程度かかります。機械学習としては非常に高速です。

In [3]:
%%bigquery
CREATE OR REPLACE MODEL
    serverlessml.model1_rawdata

OPTIONS(input_label_cols=['fare_amount'],
        model_type='linear_reg') AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count * 1.0 AS passengers
FROM
    `nyc-tlc.yellow.trips`
WHERE
    ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1

Query complete after 0.01s: 100%|██████████| 3/3 [00:00<00:00, 1060.06query/s]                        


""


訓練が完了したら、[BigQuery Cloud Console](https://console.cloud.google.com/bigquery) にアクセスして訓練されたモデルを確認してください。その後、このノートブックに戻ります。

BigQuery は提供されたデータを自動的に分割し、データの一部のみで訓練を行い、残りを評価に使用したことに注意してください。そのホールドアウトされたデータに対する評価統計を確認できます。

In [4]:
%%bigquery
SELECT * FROM ML.EVALUATE(MODEL serverlessml.model1_rawdata)

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.41s/rows]


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,6.086824,93.909516,0.32947,4.501958,0.00215,0.002582


着目している誤差である二乗平均平方根誤差（RMSE）のみをレポートしましょう。

In [5]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model1_rawdata)

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.38s/rows]


,rmse
0,9.690692


前述のとおり、あまり良い結果ではありませんでした。ヒューリスティックでは 8.13 が得られ、目標は 6 ドルであることを思い出してください。

誤差は評価対象のデータセットに依存することに注意してください。
独自のホールドアウトされたベンチマーク / テストデータセットでモデルを評価することもできますが、これを習慣にしてはいけません（ベンチマークデータセットは最終評価として保持し、プロセス全体を通じて判断を下すために使用しないようにする必要があります。そうしてしまうと、テストデータセットが真に独立したものにならないためです）。

In [6]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model1_rawdata, (
    SELECT
        (tolls_amount + fare_amount) AS fare_amount,
        pickup_longitude AS pickuplon,
        pickup_latitude AS pickuplat,
        dropoff_longitude AS dropofflon,
        dropoff_latitude AS dropofflat,
        passenger_count * 1.0 AS passengers
    FROM
        `nyc-tlc.yellow.trips`
    WHERE
        ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 2
        AND trip_distance > 0
        AND fare_amount >= 2.5
        AND pickup_longitude > -78
        AND pickup_longitude < -70
        AND dropoff_longitude > -78
        AND dropoff_longitude < -70
        AND pickup_latitude > 37
        AND pickup_latitude < 45
        AND dropoff_latitude > 37
        AND dropoff_latitude < 45
        AND passenger_count > 0
    ))

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.17s/rows]


,rmse
0,9.428081


## モデル 2: データのクリーンアップの適用

前のラボでいくつかのデータクリーンアップを行ったことを思い出してください。訓練の前にそれらを実行しましょう。

これはこのノートブックで頻繁に必要となるデータセットであるため、最初に抽出しておきます。

In [7]:
%%bigquery
CREATE OR REPLACE TABLE
    serverlessml.cleaned_training_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers
FROM
    `nyc-tlc.yellow.trips`
WHERE
    ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1
    AND trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0

Query complete after 0.00s: 100%|██████████| 3/3 [00:00<00:00, 1514.55query/s]                        


""


In [8]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在することを確認できます。
SELECT * FROM serverlessml.cleaned_training_data
LIMIT 0

Query complete after 0.00s: 100%|██████████| 1/1 [00:00<00:00, 490.73query/s]                          
Downloading: 0rows [00:00, ?rows/s]


,fare_amount,pickuplon,pickuplat,dropofflon,dropofflat,passengers


In [9]:
%%bigquery
CREATE OR REPLACE MODEL
    serverlessml.model2_cleanup

OPTIONS(input_label_cols=['fare_amount'],
        model_type='linear_reg') AS

SELECT
    *
FROM
    serverlessml.cleaned_training_data

Query complete after 0.00s: 100%|██████████| 3/3 [00:00<00:00, 1230.60query/s]                        


""


In [10]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model2_cleanup)

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.50s/rows]


,rmse
0,7.919123


## モデル 3: より高度なモデル

より高度なモデルを試すとどうなるでしょうか。BigQuery でディープニューラルネットワーク（DNN）を試してみましょう。

### DNN
DNN を作成するには、model_type に __dnn_regressor__ を指定し、隠れ層（hidden layers）を追加するだけです。

In [11]:
%%bigquery
-- この訓練には 15 分程度かかります。
CREATE OR REPLACE MODEL
    serverlessml.model3b_dnn

OPTIONS(input_label_cols=['fare_amount'],
        model_type='dnn_regressor', hidden_units=[32, 8]) AS

SELECT
    *
FROM
    serverlessml.cleaned_training_data

Query complete after 0.00s: 100%|██████████| 3/3 [00:00<00:00, 1615.26query/s]                        


""


In [12]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model3b_dnn)

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.39s/rows]


,rmse
0,4.680998


素晴らしい結果です。

## ベンチマークデータセットでの DNN の評価

同じ検証データセットを使用して評価しましょう。評価指標はデータセットに依存することを忘れないでください。同じホールドアウトデータで実行しない限り、2 つのモデルを比較することはできません。

In [13]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model3b_dnn, (
    SELECT
        (tolls_amount + fare_amount) AS fare_amount,
        pickup_datetime,
        pickup_longitude AS pickuplon,
        pickup_latitude AS pickuplat,
        dropoff_longitude AS dropofflon,
        dropoff_latitude AS dropofflat,
        passenger_count * 1.0 AS passengers,
        'unused' AS key
    FROM
        `nyc-tlc.yellow.trips`
    WHERE
        ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 10000)) = 2
        AND trip_distance > 0
        AND fare_amount >= 2.5
        AND pickup_longitude > -78
        AND pickup_longitude < -70
        AND dropoff_longitude > -78
        AND dropoff_longitude < -70
        AND pickup_latitude > 37
        AND pickup_latitude < 45
        AND dropoff_latitude > 37
        AND dropoff_latitude < 45
        AND passenger_count > 0
    ))

Downloading: 100%|██████████| 1/1 [00:01<00:00,  1.14s/rows]


,rmse
0,6.163327


素晴らしい結果です。この一連のノートブックの後半では 4 ドル未満を達成しますが、わずかな労力でこれだけの結果が得られるのは非常に優れています。

このノートブックでは、BigQuery ML を使用して ML モデルを迅速に構築する方法を紹介しました。さまざまなタイプの特徴量エンジニアリングを実験したいときに、再び BigQuery ML を使用します。BigQuery ML のスピードは、開発において非常に魅力的です。

Copyright 2019 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.